In [1]:
import json
import re
from pathlib import Path
from typing import Any
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

C:\2026-Projects\Document_Intelligent_Hub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

CHUNKED_DOCUMENTS_PATH = PROCESSED_DATA_DIR / "chunked_documents.json"
FLAGGED_RISKS_PATH = PROCESSED_DATA_DIR / "flagged_risks.json"

print(f"Chunked imput: {CHUNKED_DOCUMENTS_PATH}")
print(f"Flagged risks: {FLAGGED_RISKS_PATH}")

Chunked imput: C:\document_hub\notebooks\data\processed\chunked_documents.json
Flagged risks: C:\document_hub\notebooks\data\processed\flagged_risks.json


In [3]:
def json_records_to_documents(records: list[dict[str, Any]]) -> list[Document]:
    documents: list[Document] = []
    for record in records:
        documents.append(
            Document(
                page_content=record["page_content"],
                metadata=record["metadata"],
            )
        )
    return documents

if not CHUNKED_DOCUMENTS_PATH.exists():
    raise FileNotFoundError(
        f"Missing chunked documents at {CHUNKED_DOCUMENTS_PATH}. Run 02_chunking.ipynb first. "
    )
with CHUNKED_DOCUMENTS_PATH.open("r", encoding="utf-8") as f:
    chunked_documents_records = json.load(f)

chunked_documents = json_records_to_documents(chunked_documents_records)
print(f"Loaded {len(chunked_documents)} chunked documents.")

Loaded 5 chunked documents.


In [4]:
if chunked_documents:
    print("Simple metadata preview:")
    print(chunked_documents[0].metadata)
    print("\nSample content preview:")
    print(chunked_documents[0].page_content[:800])

Simple metadata preview:
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'moddate': '2026-06-20T21:35:22+02:00', 'source': 'Sample Compliance Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'document_id': 'a7777030-0f2d-4a90-a97e-c92105894b71', 'file_path': 'C:\\document_hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'file_type': 'pdf', 'department': 'Compliance', 'access_role': 'Compliance Analyst', 'ingested_at': '2026-06-26T16:46:03.274307+00:00', 'page_number': 1, 'part_index': 0, 'chunk_id': '5b8ab536-fd07-45ad-9e2e-fba16b375fbd', 'chunk_index': 0, 'chunk_size': 991}

Sample content preview:
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contr

In [5]:
RISK_CATEGORIES = [
    "AML",
    "KYC",
    "Privacy",
    "Data Retention",
    "Sanctions",
    "Cybersecurity",
    "Financial Reporting",
    "Third-Party Risk",
    "Legal Liability",
    "Operational Risk"
]

SEVERITY_LEVELS = ["Low", "Medium", "High", "Critical"]

RISK_SCHEMA_TEMPLATE = {
    "risk_category": "AML",
    "severity": "High",
    "clause": "Transactions above the threshold must be reviewed...",
    "reason": "Clause creates AML escalation obligation.",
    "recommended_action": "Validate threshold against current AML policy.",
    "source_document": "aml_policy.pdf",
    "page": 4,
    "chunk_id": "abc-123"
}

In [6]:
llm = ChatOllama(
    model="qwen3:8b",
    temperature=0,
)

risk_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a compliance risk extraction assistant.
            Task:
            -Analyze the provided document chunk.
            -Identify explicit compliance risks.
            -Return ONLY a JSON array.
            -If no clear risk exists, return an empty array.

            Allowed risk categories:
            {risk_categories}

            Allowed severity levels:
            {severity_levels}

            Rules:
            1) DO NOT invent facts beyond the chunk.
            2) Keep clause as a short direct quote/snippet from the chunk when possible.
            3) Keep reason concise and compliance-focused.
            4) recommended_action must be practical and actionable.
            5) Output must be valid JSON array of objects.
            6) Every object must contain:
            - risk_category
            - severity
            - clause
            - reason
            - recommended_action
            - source_document
            - page
            - chunk_id
            """.strip(),
        ),
        (
            "human",
            """
        Chunk metadata:
        -source_document: {source_document}
        -page: {page}
        -chunk_id: {chunk_id}

        Chunk text:
        {chunk_text}

        Return JSON array only.
            """.strip(),
        ),
    ]
)

risk_chain = risk_prompt | llm | StrOutputParser()

In [7]:
def _extract_json_array(raw_text: str) -> list[dict[str, Any]]:
    """"
    Parse model output robustly:
    -Supports plain JSON arrays
    -Supports fenced code blocks
    -Falls back to extracting first [...] block
    """
    text = raw_text.strip()

    if text.startswith("```json"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text).strip()
    try:
        data = json.loads(text)
        return data if isinstance(data, list) else []
    except json.JSONDecodeError:
        pass

    match = re.search(r"\[.*?\]", text, flags=re.DOTALL)
    if not match:
        return []
    try:
        data = json.loads(match.group(0))
        return data if isinstance(data, list) else []
    except json.JSONDecodeError:
        return []

In [8]:
def _normalize_severity(severity: str) -> str:
    if not severity:
        return "Medium"

    sev = severity.strip().lower()
    mapping = {
        "low": "Low",
        "medium": "Medium",
        "high": "High",
        "critical": "Critical",
        "severe": "Critical",
    }
    return mapping.get(sev, "Medium")

def _normalize_category(category: str) -> str:
    if not category:
        return "Operational Risk"

    clean = category.strip().lower()
    lookup = {c.lower(): c for c in RISK_CATEGORIES}
    return lookup.get(clean, "Operational Risk")

def validate_and_normalize_risks(
        risks: list[dict[str, Any]],
        source_document: str,
        page: Any,
        chunk_id: str,

) -> list[dict[str, Any]]:
    validated: list[dict[str, Any]] = []

    for risk in risks:
        if not isinstance(risk, dict):
            continue

        item = {
            "risk_category": _normalize_category(str(risk.get("risk_category", ""))),
            "severity": _normalize_severity(risk.get("severity", "")),
            "clause": str(risk.get("clause", "")).strip(),
            "reason": str(risk.get("reason", "")).strip(),
            "recommended_action": str(risk.get("recommended_action", "")).strip(),
            "source_document": source_document,
            "page": risk.get("page", page),
            "chunk_id": str(risk.get("chunk_id", "")).strip() or chunk_id,
        }

        if not item["clause"] or not item["reason"]:
            continue
        validated.append(item)

    return validated

In [9]:
def extract_risks_from_chunk(document: Document) -> list[dict[str, Any]]:
    source_document = document.metadata.get("source", "Unknown_source")
    page = document.metadata.get("page", "N/A")
    chunk_id = document.metadata.get("chunk_id", "N/A")
    chunk_text = document.page_content

    raw_output = risk_chain.invoke(
        {
            "risk_categories": ", ".join(RISK_CATEGORIES),
            "severity_levels": ", ".join(SEVERITY_LEVELS),
            "source_document": source_document,
            "page": page,
            "chunk_id": chunk_id,
            "chunk_text": chunk_text,
        }
    )

    parsed_risks = _extract_json_array(raw_output)
    validated_risks = validate_and_normalize_risks(
        risks = parsed_risks,
        source_document = source_document,
        page = page,
        chunk_id = chunk_id,
    )
    return validated_risks

In [10]:
def deduplicate_risks(risks: list[dict[str, Any]]) -> list[dict[str, Any]]:
    seen: set[tuple[str, str, str, str]] = set()
    unique: list[dict[str, Any]] = []

    for risk in risks:
        key = (
            risk.get("risk_category", ""),
            risk.get("severity", ""),
            risk.get("clause", "").strip().lower(),
            risk.get("source_document", ""),
        )

        if key in seen:
            continue
        seen.add(key)
        unique.append(risk)

    return unique

In [12]:
SMOKE_TEST_CHUNKS = 5
smoke_risks: list[dict[str, Any]] = []

for i, doc in enumerate(chunked_documents[:SMOKE_TEST_CHUNKS], start=1):
    print(f"Processing smoke chunk {i}/{SMOKE_TEST_CHUNKS}...")
    chunk_risks = extract_risks_from_chunk(doc)
    smoke_risks.extend(chunk_risks)

smoke_risks = deduplicate_risks(smoke_risks)
print(f"\nSmoke test risk found: {len(smoke_risks)}")

for idx, risk in enumerate(smoke_risks[:10], start=1):
    print("-" * 100)
    print(f"Risk {idx}: [{risk['severity']} {risk['risk_category']}]")
    print(f"Source: {risk['source_document']} | Page: {risk['page']}")
    print(f"Clause: {risk['clause'][:220]}")
    print(f"Reason: {risk['reason']}")
    print(f"Recommended Action: {risk['recommended_action']}")

Processing smoke chunk 1/5...
Processing smoke chunk 2/5...


KeyboardInterrupt: 

In [13]:
RUN_FULL_EXTRACTION = True #set to False to skip full extraction

all_risks: list[dict[str, Any]] = []

if RUN_FULL_EXTRACTION:
    total = len(chunked_documents)
    for i, doc in enumerate(chunked_documents, start=1):
        if i % 10 == 0 or i == total:
            print(f"Processing chunk {i}/{total}...")
        all_risks.extend(extract_risks_from_chunk(doc))

    all_risks = deduplicate_risks(all_risks)
    print(f"\nTotal unique risks found: {len(all_risks)}")
else:
    all_risks = smoke_risks

KeyboardInterrupt: 

In [ ]:
if all_risks:
    by_category: dict[str, list[dict[str, Any]]] = {}
    by_severity: dict[str, list[dict[str, Any]]] = {}

    for r in all_risks:
        by_category[r["risk_category"]] = by_category.get(r["risk_category"], 0) + 1
        by_severity[r["severity"]] = by_severity.get(r["severity"], 0) + 1

    print("\nRisk counts by severity:")
    for sev in ["Critical", "High", "Medium", "Low"]:
        print(f"- {sev}: {by_severity.get(sev, 0)}")
else:
    print("No risks extracted.")

In [ ]:
with FLAGGED_RISKS_PATH.open("w", encoding="utf-8") as f:
    json.dump(all_risks, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_risks)} risks to {FLAGGED_RISKS_PATH}")

In [ ]:
with FLAGGED_RISKS_PATH.open("r", encoding="utf-8") as f:
    reloaded_risks = json.load(f)

print(f"Reloaded {len(reloaded_risks)} risks from {FLAGGED_RISKS_PATH}")

if reloaded_risks:
    print("\nFirst risk recorded:")
    print(json.dumps(reloaded_risks[0], indent=2, ensure_ascii=False))